In [ ]:
import numpy as np
import cupy as cp
from scipy import ndimage
from scipy.fft import fftn, ifftn, fftshift, ifftshift,fft2,ifft2
from holotomocupy.utils import *


In [ ]:
n = 128
ntheta = 360
detector_pixelsize = 1.4760147601476e-6 * 2 * 2048/n
energy = 17.1
wavelength = 1.24e-09 / energy
focustodetectordistance = 1.217

sx0 = -3.135e-3
z1 = np.array([5.110, 5.464, 6.879, 9.817, 10.372, 11.146, 12.594, 17.209]) * 1e-3 - sx0
z1_ids = np.array([0, 1, 2, 3]) ### note using index starting from 0
z1 = z1[z1_ids]
ndist = len(z1)
z2 = focustodetectordistance - z1
distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]

nobj = int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 
theta = cp.linspace(0,np.pi,ntheta).astype('float32')


In [ ]:
def gen_object(n,delta,beta):
    """Generate a 3d-standard-like sample with layers, for given delta and beta scalings"""

    obj = np.zeros((n, n, n), dtype=np.float32)

    rr = (np.ones(8) * n * 0.2).astype(np.int32)
    amps = np.array([3, -3, 1, 3, -4, 1, 4], dtype=np.float32)
    dil  = (np.array([33, 28, 25, 21, 16, 10, 3], dtype=np.float32) / 256.0) * n

    # Precompute x,y,z and r^2 once
    ax = np.arange(-n//2, n//2, dtype=np.float32)
    x, y, z = np.meshgrid(ax, ax, ax, indexing="ij")
    r2 = x*x + y*y + z*z

    # frame edges without swapaxes (same geometry)
    def frame_edges(p1, p2):
        c = np.zeros((n, n, n), dtype=np.float32)
        c[p1:p2, p1, p1] = 1; c[p1:p2, p1, p2] = 1
        c[p1:p2, p2, p1] = 1; c[p1:p2, p2, p2] = 1
        c[p1, p1:p2, p1] = 1; c[p1, p1:p2, p2] = 1
        c[p2, p1:p2, p1] = 1; c[p2, p1:p2, p2] = 1
        c[p1, p1, p1:p2] = 1; c[p1, p2, p1:p2] = 1
        c[p2, p1, p1:p2] = 1; c[p2, p2, p1:p2] = 1
        return c

    fcirc_list = []
    for d in dil:
        circ = (r2 < (d*d)).astype(np.float32)
        fcirc_list.append((fftn(fftshift(circ))))

    for kk, a in enumerate(amps):
        r = int(rr[kk])
        p1 = n//2 - r//2
        p2 = n//2 + r//2

        cube = frame_edges(p1, p2)

        # Match original cube FFT:
        fcube = (fftn(fftshift(cube)))
        conv  = fftshift(ifftn((fcube * fcirc_list[kk]))).real

        obj += a * (conv > 1.0).astype(np.float32)

    obj = ndimage.rotate(obj, 28, axes=(0, 1), reshape=False, order=1)
    obj = ndimage.rotate(obj, 45, axes=(0, 2), reshape=False, order=1)    
    obj = np.roll(obj, -15*n//256, axis=2)
    obj = np.roll(obj, -10*n//256, axis=1)

    v = (np.arange(-n//2, n//2, dtype=np.float32) / n)
    vx, vy, vz = np.meshgrid(v, v, v, indexing="ij")
    filt = fftshift(np.exp(-10.0 * (vx*vx + vy*vy + vz*vz)).astype(np.float32))

    fu = fftn((obj))
    obj = ifftn((fu * filt)).real

    obj[obj < 0] = 0
    return (obj * (-delta + 1j*beta) ).astype(np.complex64)

obj = gen_object(nobj,1,1e-2)
mshow_complex(obj[:, nobj//2],True)
mshow_complex(obj[nobj//2],True)

In [ ]:
def read_probe(n,ndist):
    !wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_abs_2048.tiff -P data/prb_id16a
    !wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_phase_2048.tiff -P data/prb_id16a

    prb_abs = read_tiff(f'data/prb_id16a/prb_abs_2048.tiff')[:ndist]
    prb_phase = read_tiff(f'data/prb_id16a/prb_phase_2048.tiff')[:ndist]
    prb = prb_abs*np.exp(1j*prb_phase).astype('complex64')


    for k in range(int(np.log2(2048/n))):
        prb = prb[:, ::2]+prb[:, 1::2]
        prb = prb[:, :, ::2]+prb[:, :, 1::2]/4

    prb /= np.mean(np.abs(prb))
    prb = np.pad(prb[:,n//4:-n//4,n//4:-n//4],((0,0),(n//4,n//4),(n//4,n//4)),'symmetric')

    v = (np.arange(-n//2, n//2, dtype=np.float32) / n)
    vx, vy = np.meshgrid(v, v, indexing="ij")
    filt = fftshift(np.exp(-4.0 * (vx*vx + vy*vy)).astype(np.float32))

    fu = fft2((prb))
    prb = ifft2((fu *filt))

    
    return prb

prb = read_probe(n,ndist)
mshow_polar(prb[0],True)
mshow_polar(prb[-1],True)

In [ ]:
pos = ((np.random.random([ntheta,ndist,2])-0.5)*n/8).astype('float32')

mshow_pos(pos,True)


In [ ]:
np.save(f'data/obj_{n}',obj)
np.save(f'data/prb_{n}',prb)
np.save(f'data/pos_{n}',pos)
